# Adaptive Router

Bootstrap the notebook so project imports work even when Jupyter starts in the `notebooks/` folder.

In [ ]:
from pathlib import Path
import sys
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")
print(PROJECT_ROOT)

### Initialize Models and Imports

In [ ]:
from src.adaptive_router import llm_router
from src.config import parent_store_collection
from src.generation.generator import generate_answer
from src.retrieval.retriever import search_vector_db
from src.caching.semantic_cache import get_semantic_cache, set_semantic_cache
from langchain_groq import ChatGroq
from src.config import OPENAI_MODEL_GROQ, TEMPERATURE, MAX_TOKENS, GROQ_API_KEY

llm = ChatGroq(
    model=OPENAI_MODEL_GROQ,
    api_key=GROQ_API_KEY,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)


### Full-Fledged Adaptive Routing Flow
This logic determines the route based on the query and executes the appropriate pipeline.

In [ ]:
def process_query(query: str, namespace: str = "default_namespace"):
    print(f"\nQuery: '{query}'")
    
    route = llm_router(query)
    print(f"--- Route Selected: {route} ---")
    
    if route == "vectorstore":
        cached_answer, cached_sources, query_emb = get_semantic_cache(query=query)
        
        if cached_answer:
            print("--- SEMANTIC CACHE HIT ---")
            print(cached_answer)
            return

        print("--- CACHE MISS. RETRIEVING & GENERATING ---")
        retrieved_chunks = search_vector_db(namespace=namespace, query=query, top_k=5)
        
        if retrieved_chunks:
            answer = generate_answer(query, retrieved_chunks, namespace)
            print("\n--- LLM ANSWER ---")
            print(answer)
            
            sources = list(set([chunk.get("source") for chunk in retrieved_chunks]))
            set_semantic_cache(query=query, answer=answer, sources=sources, query_emb=query_emb)
        else:
            print("No relevant context found to answer the query.")
            
    elif route == "llm_direct":
        print("--- EXECUTING DIRECT LLM GENERATION ---")
        answer = llm.invoke(query).content
        print("\n--- LLM ANSWER ---")
        print(answer)
        
    elif route == "web_search":
        print("--- EXECUTING WEB SEARCH FLOW ---")
        print("Web search API not integrated yet")
        
    else:
        print(f"just in case llm hallucinates: {route}")

### Test the Router

In [ ]:
# Test 1: General Knowledge (Should route to llm_direct)
process_query("What is the capital of France?")
print("\n" + "="*50 + "\n")

# Test 2: RAG Context (Should route to vectorstore)
process_query("What is the main topic of the document and explain its chunking strategy?")
print("\n" + "="*50 + "\n")

# Test 3: Web Search (Should route to web_search)
process_query("What is Apple's stock price today?")
